In [3]:
import networkx as nx
from itertools import combinations

In [4]:
def parte1_validar(G):
    print("=" * 70)
    print("PARTE 1 — Validação do grafo")
    print("=" * 70)
    eh_dag = nx.is_directed_acyclic_graph(G)
    print(f"  É acíclico (DAG válido)? {eh_dag}")
    if not eh_dag:
        ciclo = nx.find_cycle(G)
        print(f"  >> CICLO encontrado: {ciclo}  (precisa ser removido)")
        return False
    print(f"  Nós ({G.number_of_nodes()}): {sorted(G.nodes())}")
    print(f"  Arestas: {G.number_of_edges()}")
    print(f"  Ordem topológica: {list(nx.topological_sort(G))}")
    return True

In [5]:
def parte2_refutabilidade(G):
    print("\n" + "=" * 70)
    print("PARTE 2 — Refutabilidade (implicações testáveis via d-separação)")
    print("=" * 70)

    def adjacentes(a, b):
        return G.has_edge(a, b) or G.has_edge(b, a)

    implicacoes = []
    for x, y in combinations(sorted(G.nodes()), 2):
        if adjacentes(x, y):
            continue  # adjacentes nunca são d-separáveis -> não geram teste
        z = nx.find_minimal_d_separator(G, x, y)  # conjunto mínimo de separação
        if z is not None:
            implicacoes.append((x, y, set(z)))

    if not implicacoes:
        print("  Nenhuma independência implicada -> grafo SATURADO -> IRREFUTÁVEL.")
        return implicacoes

    print(f"  O DAG é REFUTÁVEL: implica {len(implicacoes)} independências testáveis.\n")
    for x, y, z in implicacoes:
        cond = ", ".join(sorted(z)) if z else "(nenhuma — independência marginal)"
        print(f"    {x}  _||_  {y}   |  {cond}")
    print("\n  -> Teste cada uma nos seus dados; se alguma for violada, o DAG está errado.")
    return implicacoes

In [6]:
def conjunto_ajuste_backdoor(G, T, Y):
    if T not in G or Y not in G:
        return None
    if Y in (nx.descendants(G, T) | {T}):
        permitidos = set(G.nodes()) - (nx.descendants(G, T) | {T})
    else:
        # T não é causa de Y; efeito causal total é zero por construção
        return None
    Gb = G.copy()
    Gb.remove_edges_from(list(G.out_edges(T)))   # grafo de back-door
    try:
        return nx.find_minimal_d_separator(Gb, T, Y, restricted=permitidos)
    except nx.NetworkXError:
        return None


def parte3_identificabilidade(G, Y, tratamentos):
    print("\n" + "=" * 70)
    print("PARTE 3 — Identificabilidade do efeito sobre", Y)
    print("=" * 70)
    latentes = False  # este DAG é totalmente observado
    print("  Há variáveis latentes / confundimento não medido? ", "Sim" if latentes else "Não")
    print("  => Sem latentes, TODO efeito causal é IDENTIFICÁVEL por ajuste (back-door).\n")
    print("  Conjunto de ajuste mínimo (controlar por estas variáveis dá o efeito total):")
    for T in tratamentos:
        if T == Y:
            continue
        Z = conjunto_ajuste_backdoor(G, T, Y)
        if Z is None:
            print(f"    {T:>20s} -> {Y}:  (T não é causa de {Y} neste DAG)")
        else:
            ajuste = ", ".join(sorted(Z)) if Z else "(conjunto vazio — sem confundidores)"
            print(f"    {T:>20s} -> {Y}:  ajustar por {{ {ajuste} }}")

In [7]:
"""
==============================================================================
Verificação de um DAG causal: é "respondível" (identificável) e "refutável"?
Y (desfecho) = morte_evitavel
==============================================================================

"""

ARESTAS = [
    # raca_cor ->
    ("raca_cor", "sigla_uf"),
    ("raca_cor", "idade_grupo"),
    ("raca_cor", "ocupacao"),
    ("raca_cor", "morte_evitavel"),
    # sexo ->
    ("sexo", "idade_grupo"),
    ("sexo", "escolaridade_grupo"),
    ("sexo", "ocupacao"),
    ("sexo", "morte_evitavel"),
    # sigla_uf ->
    ("sigla_uf", "ocupacao"),
    ("sigla_uf", "local_ocorrencia"),
    ("sigla_uf", "morte_evitavel"),
    # idade_grupo ->
    ("idade_grupo", "escolaridade_grupo"),
    ("idade_grupo", "ocupacao"),
    ("idade_grupo", "local_ocorrencia"),
    ("idade_grupo", "morte_evitavel"),
    # ano ->
    ("ano", "escolaridade_grupo"),
    ("ano", "ocupacao"),
    ("ano", "morte_evitavel"),
    # escolaridade_grupo ->
    ("escolaridade_grupo", "ocupacao"),
    ("escolaridade_grupo", "morte_evitavel"),
    # ocupacao ->
    ("ocupacao", "local_ocorrencia"),
    ("ocupacao", "morte_evitavel"),
    # local_ocorrencia ->
    ("local_ocorrencia", "morte_evitavel"),
]

DESFECHO = "morte_evitavel"   # Y
TRATAMENTOS = ["raca_cor", "escolaridade_grupo", "ocupacao", "sigla_uf", "ano"]

G = nx.DiGraph(ARESTAS)


if __name__ == "__main__":
    if parte1_validar(G):
        parte2_refutabilidade(G)
        parte3_identificabilidade(G, DESFECHO, TRATAMENTOS)
        

PARTE 1 — Validação do grafo
  É acíclico (DAG válido)? True
  Nós (9): ['ano', 'escolaridade_grupo', 'idade_grupo', 'local_ocorrencia', 'morte_evitavel', 'ocupacao', 'raca_cor', 'sexo', 'sigla_uf']
  Arestas: 23
  Ordem topológica: ['raca_cor', 'sexo', 'ano', 'sigla_uf', 'idade_grupo', 'escolaridade_grupo', 'ocupacao', 'local_ocorrencia', 'morte_evitavel']

PARTE 2 — Refutabilidade (implicações testáveis via d-separação)
  O DAG é REFUTÁVEL: implica 13 independências testáveis.

    ano  _||_  idade_grupo   |  (nenhuma — independência marginal)
    ano  _||_  local_ocorrencia   |  idade_grupo, ocupacao, sigla_uf
    ano  _||_  raca_cor   |  (nenhuma — independência marginal)
    ano  _||_  sexo   |  (nenhuma — independência marginal)
    ano  _||_  sigla_uf   |  (nenhuma — independência marginal)
    escolaridade_grupo  _||_  local_ocorrencia   |  idade_grupo, ocupacao, sigla_uf
    escolaridade_grupo  _||_  raca_cor   |  idade_grupo, sexo
    escolaridade_grupo  _||_  sigla_uf   |  i